![logo](../images/logo_diive1_128px.png)

<span style='font-size:40px; display:block;'><b>Download specific variables from database (influxdb)</b></span>

---
**Notebook version**: `1` (16 Jul 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

# ℹ️ About this notebook

Download one or more specific variables from the InfluxDB database into a pandas DataFrame, take a quick look, and (optionally) save them to a file. Downloading uses diive's in-house InfluxDB engine ([`InfluxIO`](../diive/core/io/db/influx/influxio.py), in `diive/core/io/db/influx`), which needs only `uv sync --group db`.

This is a **read-only** notebook: it downloads and inspects data, it does not screen, correct, or upload anything. For quality screening with an upload back to the database, use [DatabaseInfluxStepwiseMeteoScreening.ipynb](DatabaseInfluxStepwiseMeteoScreening.ipynb).

# ⏱️ Timestamp convention

The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`** (the stamp marks the *end* of the averaging interval). On download, `InfluxIO` shifts the UTC timestamp by `TIMEZONE_OFFSET_TO_UTC_HOURS`, so the returned data are in **local time** (UTC + offset) and still `TIMESTAMP_END`.

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |

Set `TIMEZONE_OFFSET_TO_UTC_HOURS` to the timezone you want the data returned in (e.g. `1` for CET winter time). `START` and `STOP` are interpreted in that same timezone.

# ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Source**
- `SITE`: site ID, used to build the bucket name.
- `BUCKET`: the InfluxDB bucket to read from. By convention `{SITE}_raw` holds raw data and `{SITE}_processed` holds processed/screened data.
- `DATA_VERSION`: version ID of the data to download, e.g. `'raw'` or `'meteoscreening_diive'`. `None` downloads across all versions.

**Variables to download**
- `MEASUREMENTS`: measurement group(s), e.g. `['TA', 'SW', 'RH']`.
- `FIELDS`: variable name(s) exactly as stored in the database (the InfluxDB `_field`), e.g. `['TA_T1_2_1', 'SW_IN_T1_2_1']`.

**Time range**
- `START`: first timestamp to download — **is** included.
- `STOP`: upper bound — **is not** included.

**Timestamp / config**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: timezone the data is returned in — see the *Timestamp convention* section above (e.g. `1` for CET winter time).
- `DIRCONF`: local folder holding the database connection config.

In [ ]:
# --- Source ---
SITE = 'ch-cha'
BUCKET = f'{SITE}_processed'  # source bucket, e.g. 'ch-cha_processed' (use '{SITE}_raw' for raw data)
DATA_VERSION = 'meteoscreening_diive'  # e.g. 'raw', 'meteoscreening_diive', or None for all versions

# --- Variables to download ---
MEASUREMENTS = ['TA', 'SW', 'RH']
FIELDS = ['TA_T1_2_1', 'SW_IN_T1_2_1', 'RH_T1_2_1']

# --- Time range ---
START = '2022-04-01 00:00:01'  # included
STOP = '2022-04-03 00:00:01'  # not included

# --- Timestamp / config ---
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time)
DIRCONF = r'F:\dev\poet\configs'  # <-- set to your config folder
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

## Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import pandas as pd

import diive as dv
from diive.core.io.db.influx import InfluxIO  # diive's in-house InfluxDB engine (needs: uv sync --group db)

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
print(f"Last run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

# 🔌 Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional — list all fields available in a measurement (does not check the selected time range):

In [ ]:
# display(dbc.show_fields_in_measurement(bucket=BUCKET, measurement=MEASUREMENTS[0]))

# ⬇️ Download data from database
Returns three objects:
- `data_simple`: high-res time series, one column per variable (nice to look at).
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series **and its database tags**.
- `assigned_measurements`: the auto-detected measurement per variable (a sanity check).

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET,
    measurements=MEASUREMENTS,
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

## Inspect downloaded data

In [ ]:
data_simple

In [ ]:
assigned_measurements

## Verify download timestamps
Confirm the timestamps look right: they should be in **local time** (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`) and mark the **end** of each averaging interval. Eyeball the first/last stamps against the `START`/`STOP` you requested.

In [ ]:
for v in data_detailed.keys():
    idx = data_detailed[v].index
    print(f'{v}: index name={idx.name!r}, tz={idx.tz}, freq={idx.freqstr}')
    print(f'   first={idx[0]}   last={idx[-1]}')
print(f'\nApplied UTC offset: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h (timestamps above are local time)')

# 📈 Plot downloaded data

In [ ]:
data_simple.plot(x_compat=True, figsize=(16, 6), title='Downloaded data', subplots=True);

# 💾 Save to file (optional)
Uncomment one of the lines below to write the downloaded data to disk. Left commented so a top-to-bottom *Run All* does not create files.

In [ ]:
# data_simple.to_csv('data.csv')                         # simple CSV, one column per variable
# dv.save_parquet(filename='data', data=data_simple, outpath='.')  # diive parquet

# ✅ End of notebook

In [ ]:
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")